# Step 8 — MPS SLAM Validation (fixed): exactly the 120 experiment recordings

Replaces notebook 04. Differences: matches the MPS folder for each of the **120 experiment trials** (handling the three capitalized `mps_Ofire0_*` frontal-office folders automatically — no renaming needed), excludes the 9 pilots, and reports the summary that backs Table 5's caption.

**Outputs:** `results/mps_results.csv`, `results/mps_summary.json`. Paste the printed summary block back to Claude.

In [ ]:
from pathlib import Path

# ── Base path Set your base path here ─────────────────
BASE = Path('/path/to/your/esas_project')  # <-- set this
# ────────────────────────────────────────────────────────────
RECORDINGS  = BASE / 'recordings'
RESULTS_DIR = Path('results'); RESULTS_DIR.mkdir(exist_ok=True)
print(f'Recordings dir exists: {RECORDINGS.exists()}')

In [ ]:
import csv, math
import numpy as np
import pandas as pd

def quat_to_yaw(qx, qy, qz, qw):
    siny = 2*(qw*qz + qx*qy)
    cosy = 1 - 2*(qy*qy + qz*qz)
    return math.degrees(math.atan2(siny, cosy))

# The 120 experiment stems (same design as notebook 07)
ANGLES, TRIALS = [0, 45, -45, 90, -90], (1, 2, 3)
stems  = [f'{p}{s}{a}_{t}' for p in 'ok' for s in ['fire','phon'] for a in ANGLES for t in TRIALS]
stems += [f'kbaby{a}_{t}' for a in ANGLES for t in TRIALS]
stems += [f'ccar_{a}_{t}' for a in ANGLES for t in TRIALS]
stems += [f'studio_{s}{a}_{t}' for s in ['fire','phone'] for a in ANGLES for t in TRIALS]
assert len(stems) == 120

def mps_folder(stem):
    cands = [RECORDINGS / f'mps_{stem}_vrs']
    if stem.startswith('ofire0_'):                      # capitalized legacy name
        cands.append(RECORDINGS / f'mps_O{stem[1:]}_vrs')
    for c in cands:
        if (c / 'slam' / 'open_loop_trajectory.csv').exists():
            return c
    return None

rows, missing = [], []
for stem in stems:
    d = mps_folder(stem)
    if d is None:
        missing.append(stem); continue
    df = pd.read_csv(d / 'slam' / 'open_loop_trajectory.csv', comment='#')
    q  = float(df['quality_score'].mean())
    ps = float(np.sqrt(df['tx_odometry_device'].std()**2 +
                       df['ty_odometry_device'].std()**2 +
                       df['tz_odometry_device'].std()**2))
    yaws = [quat_to_yaw(r.qx_odometry_device, r.qy_odometry_device,
                        r.qz_odometry_device, r.qw_odometry_device)
            for _, r in df.iterrows()]
    rows.append({'recording': stem, 'mps_folder': d.name, 'quality': round(q,3),
                 'pos_std_mm': round(ps*1000,2), 'yaw_std': round(float(np.std(yaws)),3),
                 'valid': q >= 0.4 and ps < 0.05})
    print(f'  {stem:<22} quality={q:.3f}  pos={ps*1000:5.2f}mm  yaw={np.std(yaws):.3f}°')

df = pd.DataFrame(rows)
print('='*54)
print('  MPS SLAM Validation — 120 experiment recordings')
print('='*54)
print(f'  Analysed:       {len(df)} / 120')
print(f'  Missing MPS:    {missing if missing else "none"}')
print(f'  Valid:          {int(df["valid"].sum())}')
print(f'  Mean quality:   {df["quality"].mean():.3f}')
print(f'  Pos stability:  {df["pos_std_mm"].mean():.2f} mm')
print(f'  Yaw stability:  {df["yaw_std"].mean():.3f}°')
print('='*54)
df.to_csv(RESULTS_DIR / 'mps_results.csv', index=False)
import json as _json
_json.dump({'analysed': len(df), 'missing': missing, 'valid': int(df['valid'].sum()),
            'mean_quality': round(float(df['quality'].mean()),3),
            'mean_pos_std_mm': round(float(df['pos_std_mm'].mean()),2),
            'mean_yaw_std_deg': round(float(df['yaw_std'].mean()),3)},
           open(RESULTS_DIR / 'mps_summary.json','w'), indent=2)
print(f'Saved: {RESULTS_DIR}/mps_results.csv and mps_summary.json')